In [ ]:
#| default_exp dataloader

## Imports and source data

Assumptions:
- `fastpragma.data` already exports `DataSource`, `Tokenizer`, and `PRAGMADataset`.
- We use MovieLens 100k as a small stand-in for banking event histories.
- User rows provide profile/static state, and rating rows provide timestamped events.

In [ ]:
#| export
from fastpragma.data import DataSource, Tokenizer, PRAGMADataset
import polars as pl
from fastai.data.external import untar_data, URLs
from fastcore.all import *

path = untar_data(URLs.ML_100k)

In [ ]:
ratings = pl.scan_csv(path/'u.data', separator='\t', has_header=False, new_columns=['user_id','movie_id','rating','timestamp'])
ratings = ratings.with_columns(pl.from_epoch('timestamp', time_unit='s').alias('timestamp'))
ratings_src = DataSource(ratings, entity_col='user_id', cats=['movie_id','rating'], time_col='timestamp', name='ratings')
ratings_src

DataSource(columns=['user_id', 'movie_id', 'rating', 'timestamp'], name=ratings cats=['movie_id', 'rating'], conts=[], texts=[], time_col='timestamp')
shape: (5, 4)
┌─────────┬──────────┬────────┬─────────────────────┐
│ user_id ┆ movie_id ┆ rating ┆ timestamp           │
│ ---     ┆ ---      ┆ ---    ┆ ---                 │
│ i64     ┆ i64      ┆ i64    ┆ datetime[μs]        │
╞═════════╪══════════╪════════╪═════════════════════╡
│ 196     ┆ 242      ┆ 3      ┆ 1997-12-04 15:55:49 │
│ 186     ┆ 302      ┆ 3      ┆ 1998-04-04 19:22:22 │
│ 22      ┆ 377      ┆ 1      ┆ 1997-11-07 07:18:36 │
│ 244     ┆ 51       ┆ 2      ┆ 1997-11-27 05:02:03 │
│ 166     ┆ 346      ┆ 1      ┆ 1998-02-02 05:33:16 │
└─────────┴──────────┴────────┴─────────────────────┘

In [ ]:
users = pl.scan_csv(path/'u.user', separator='|', has_header=False, new_columns=['user_id','age','gender','occupation','zip_code'])
users_src = DataSource(users, entity_col='user_id', cats=['gender','zip_code'], conts=['age'], texts=['occupation'], name='users', is_profile=True)
users_src

DataSource(columns=['user_id', 'age', 'gender', 'occupation', 'zip_code'], name=users cats=['gender', 'zip_code'], conts=['age'], texts=['occupation'], time_col=None)
shape: (5, 5)
┌─────────┬─────┬────────┬────────────┬──────────┐
│ user_id ┆ age ┆ gender ┆ occupation ┆ zip_code │
│ ---     ┆ --- ┆ ---    ┆ ---        ┆ ---      │
│ i64     ┆ i64 ┆ str    ┆ str        ┆ str      │
╞═════════╪═════╪════════╪════════════╪══════════╡
│ 1       ┆ 24  ┆ M      ┆ technician ┆ 85711    │
│ 2       ┆ 53  ┆ F      ┆ other      ┆ 94043    │
│ 3       ┆ 23  ┆ M      ┆ writer     ┆ 32067    │
│ 4       ┆ 24  ┆ M      ┆ technician ┆ 43537    │
│ 5       ┆ 33  ┆ F      ┆ other      ┆ 15213    │
└─────────┴─────┴────────┴────────────┴──────────┘

## Build and tokenize the dataset

Assumptions:
- `users_src` is profile state because it has one row per user and no event timestamp.
- `ratings_src` is the event stream because each row is timestamped.
- `eval_time` is fixed for this toy dataset so `logsec` features are deterministic.
- Tokenized shards plus `tokenizer.json` are saved under `out`.

In [ ]:
out = Path('data/ml100k_pragma')
ds = PRAGMADataset(profile=users_src, events=[ratings_src], entity_col='user_id', out_path=out)
ds.show_summary()

i,name,kind,rows,entities,entity_col,time_col,cats,conts,signed_conts,texts,lifelong,min_time,max_time
i64,str,str,i64,i64,str,str,i64,i64,i64,i64,i64,i64,i64
0,"""users""","""profile""",943,943,"""user_id""",null,2,1,0,1,0,null,null
1,"""ratings""","""event""",100000,943,"""user_id""","""timestamp""",2,0,0,0,0,874724710,893286638


In [ ]:
out.mkdir(exist_ok=True)
tok = ds.tokenize(eval_time='2026-01-01', n_shards=5)

Keys: 11, Vals: 2604, BPE: none


tokenizing profile


 |----------------------------------------| 0.00% [0/1 00:00<?]

tokenizing event source 0: ratings


 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

...

 |████████--------------------------------| 20.00% [1/5 00:00<00:02]

 |████████████████------------------------| 40.00% [2/5 00:01<00:01]

 |████████████████████████----------------| 60.00% [3/5 00:01<00:01]

 |████████████████████████████████--------| 80.00% [4/5 00:02<00:00]

 |████████████████████████████████████████| 100.00% [5/5 00:02<00:00]

In [ ]:
out.ls()

[Path('data/ml100k_pragma/shard_1.parquet'), Path('data/ml100k_pragma/shard_2.parquet'), Path('data/ml100k_pragma/tokenizer.json'), Path('data/ml100k_pragma/shard_3.parquet'), Path('data/ml100k_pragma/shard_0.parquet'), Path('data/ml100k_pragma/shard_4.parquet')]

## Reload saved tokenized data

Assumptions:
- The dataloader should consume saved parquet shards, not the original raw MovieLens files.
- Each shard contains rows from multiple users, with profile rows marked by `event_idx == -1` and event rows by `event_idx >= 0`.
- We first inspect one shard and one user before building batching logic.

In [ ]:
tok = Tokenizer.load(out/'tokenizer.json')
shard = pl.read_parquet(out/'shard_0.parquet')

In [ ]:
shard.shape, shard.columns, shard.head()

((45256, 15),
 ['user_id',
  'key',
  'key_id',
  'value',
  'val_id',
  'val_pos',
  'vtype',
  'time',
  'logsec',
  'source_idx',
  'event_idx',
  'hour',
  'dow',
  'dom',
  'shard_id'],
 shape: (5, 15)
 ┌─────────┬────────────┬────────┬────────┬───┬──────┬─────┬─────┬──────────┐
 │ user_id ┆ key        ┆ key_id ┆ value  ┆ … ┆ hour ┆ dow ┆ dom ┆ shard_id │
 │ ---     ┆ ---        ┆ ---    ┆ ---    ┆   ┆ ---  ┆ --- ┆ --- ┆ ---      │
 │ i64     ┆ str        ┆ i64    ┆ str    ┆   ┆ f32  ┆ f32 ┆ f32 ┆ u64      │
 ╞═════════╪════════════╪════════╪════════╪═══╪══════╪═════╪═════╪══════════╡
 │ 3       ┆ age        ┆ 5      ┆ 23     ┆ … ┆ 0.0  ┆ 0.0 ┆ 0.0 ┆ 0        │
 │ 3       ┆ gender     ┆ 6      ┆ M      ┆ … ┆ 0.0  ┆ 0.0 ┆ 0.0 ┆ 0        │
 │ 3       ┆ occupation ┆ 8      ┆ writer ┆ … ┆ 0.0  ┆ 0.0 ┆ 0.0 ┆ 0        │
 │ 3       ┆ zip_code   ┆ 10     ┆ 32067  ┆ … ┆ 0.0  ┆ 0.0 ┆ 0.0 ┆ 0        │
 │ 3       ┆ movie_id   ┆ 7      ┆ 300    ┆ … ┆ 2.0  ┆ 6.0 ┆ 7.0 ┆ 0        │
 └─────────┴─

In [ ]:
shards = sorted(out.glob('shard_*.parquet'))
len(shards), shards[:2]

(5,
 [Path('data/ml100k_pragma/shard_0.parquet'),
  Path('data/ml100k_pragma/shard_1.parquet')])

In [ ]:
sample = pl.read_parquet(shards[0]).filter(pl.col('user_id') == pl.col('user_id').first())
sample

user_id,key,key_id,value,val_id,val_pos,vtype,time,logsec,source_idx,event_idx,hour,dow,dom,shard_id
i64,str,i64,str,i64,i64,str,i64,f64,i32,i64,f32,f32,f32,u64
3,"""age""",5,"""23""",2501,0,"""num""",0,0.0,-1,-1,0.0,0.0,0.0,0
3,"""gender""",6,"""M""",2469,0,"""cat""",0,0.0,-1,-1,0.0,0.0,0.0,0
3,"""occupation""",8,"""writer""",2603,0,"""text""",0,0.0,-1,-1,0.0,0.0,0.0,0
3,"""zip_code""",10,"""32067""",1203,0,"""cat""",0,0.0,-1,-1,0.0,0.0,0.0,0
3,"""movie_id""",7,"""300""",1162,0,"""cat""",889236939,33.858346,0,0,2.0,6.0,7.0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3,"""rating""",9,"""5""",1507,0,"""cat""",889237482,0.0,0,51,2.0,6.0,7.0,0
3,"""movie_id""",7,"""317""",1196,0,"""cat""",889237482,0.0,0,52,2.0,6.0,7.0,0
3,"""rating""",9,"""2""",965,0,"""cat""",889237482,0.0,0,52,2.0,6.0,7.0,0


## Tensor helpers and limits

Assumptions:
- Each key-value token is represented as `(key_id, val_id, val_pos)`.
- Profile and lifelong tensors can be padded per batch.
- Event tokens will be packed flat with offsets instead of padded to a large rectangle.
- Cutoffs mirror the PRAGMA paper defaults where practical: profile tokens, lifelong tokens, events per user, and tokens per event.

In [ ]:
#| export
#| export
import torch
from torch.utils.data import IterableDataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

In [ ]:
#| export
max_profile_toks,max_lifelong_toks,max_events,max_event_toks,max_tokens = 200,64,6500,24,12000

In [ ]:
#| export
def _tok(df):
    if len(df)==0: return torch.empty(0,3,dtype=torch.long),torch.empty(0,dtype=torch.float32)
    return torch.tensor(df.select('key_id','val_id','val_pos').to_numpy(), dtype=torch.long),torch.tensor(df['logsec'].to_numpy(), dtype=torch.float32)

def _pad(xs, val=0):
    if len(xs)==0: return torch.empty(0,0,3,dtype=torch.long),torch.empty(0,0,dtype=torch.bool)
    lens = torch.tensor([len(o) for o in xs])
    return pad_sequence(xs, batch_first=True, padding_value=val),torch.arange(lens.max())[None,:] < lens[:,None]

In [ ]:
prof = sample.filter(pl.col('event_idx') == -1)
ev0 = sample.filter(pl.col('event_idx') == 0)

x,t = _tok(prof)
test_eq(x.shape, (len(prof),3))
test_eq(t.shape, (len(prof),))
test_eq(x.dtype, torch.long)
test_eq(t.dtype, torch.float32)

x0,t0 = _tok(ev0)
pad,mask = _pad([x, x0])
test_eq(pad.shape, (2, max(len(x),len(x0)), 3))
test_eq(mask.shape, (2, max(len(x),len(x0))))
test_eq(mask[0].sum().item(), len(x))
test_eq(mask[1].sum().item(), len(x0))

empty_x,empty_t = _tok(prof.filter(pl.col('user_id') == -1))
test_eq(empty_x.shape, (0,3))
test_eq(empty_t.shape, (0,))

## Convert one user into a record

Assumptions:
- Static profile rows have `event_idx < 0` and `time == 0`.
- Lifelong rows have `event_idx < 0` and non-zero time.
- Event rows are sorted by `event_idx`, truncated to the latest `max_events`, and each event is capped at `max_event_toks`.
- Calendar features are stored once per event.

In [ ]:
#| export
def user_rec(df):
    uid = df['user_id'][0]
    prof = df.filter((pl.col('event_idx')<0) & (pl.col('time')==0)).head(max_profile_toks)
    ll = df.filter((pl.col('event_idx')<0) & (pl.col('time')!=0)).tail(max_lifelong_toks)
    ev = df.filter(pl.col('event_idx')>=0).sort('event_idx')
    ev_ids = ev.select('event_idx').unique(maintain_order=True).tail(max_events)['event_idx'].to_list()
    ev = ev.filter(pl.col('event_idx').is_in(ev_ids))
    pt,ptm = _tok(prof)
    lt,ltm = _tok(ll)
    ets,etm,cal = [],[],[]
    for _,g in ev.partition_by('event_idx', as_dict=True, maintain_order=True).items():
        g = g.head(max_event_toks)
        t,tm = _tok(g)
        ets.append(t); etm.append(tm[0] if len(tm) else torch.tensor(0.))
        cal.append(torch.tensor(g.select('hour','dow','dom').row(0), dtype=torch.float32))
    return dict(uid=uid, profile=pt, profile_time=ptm, lifelong=lt, lifelong_time=ltm, events=ets, event_time=torch.stack(etm) if etm else torch.empty(0), cal=torch.stack(cal) if cal else torch.empty(0,3))

In [ ]:
rec = user_rec(sample)
rec.keys(), rec['uid'], rec['profile'].shape, rec['lifelong'].shape, len(rec['events']), rec['event_time'].shape, rec['cal'].shape

test_eq(rec['profile'].shape[1], 3)
test_eq(rec['profile_time'].shape, (len(rec['profile']),))
test_eq(rec['lifelong'].shape[1] if len(rec['lifelong']) else 3, 3)
test_eq(rec['lifelong_time'].shape, (len(rec['lifelong']),))
test_eq(len(rec['events']), len(rec['event_time']))
test_eq(len(rec['events']), len(rec['cal']))
test_eq(all(o.shape[1] == 3 for o in rec['events']), True)
rec['events'][0].shape, rec['event_time'][:3], rec['cal'][:3]

(torch.Size([2, 3]),
 tensor([33.8583, 33.8583, 33.8583]),
 tensor([[2., 6., 7.],
         [2., 6., 7.],
         [2., 6., 7.]]))

In [ ]:
#| export
def rec_ntoks(o): return len(o['profile']) + len(o['lifelong']) + sum(len(e) for e in o['events'])

## Pack a batch

Assumptions:
- Profiles and lifelong rows are small enough to pad within a batch.
- Event field tokens are packed into one flat tensor.
- `event_offsets` maps flat tokens to events.
- `event_user` maps events to users.
- `history_offsets` reserves one history position per user for `[USR]`, plus one per event.

In [ ]:
#| export
def pack_batch(recs):
    prof,prof_mask = _pad([o['profile'] for o in recs])
    ll,ll_mask = _pad([o['lifelong'] for o in recs])
    ptime = pad_sequence([o['profile_time'] for o in recs], batch_first=True)
    ltime = pad_sequence([o['lifelong_time'] for o in recs], batch_first=True)
    ets,eu,etime,cal,hoffs = [],[],[],[],[0]
    for i,o in enumerate(recs):
        ets += o['events']; eu += [i]*len(o['events'])
        etime.append(o['event_time']); cal.append(o['cal'])
        hoffs.append(hoffs[-1] + len(o['events']) + 1)
    lens = torch.tensor([len(o) for o in ets], dtype=torch.long)
    eoffs = torch.cat([torch.tensor([0]), lens.cumsum(0)]) if len(lens) else torch.tensor([0])
    etoks = torch.cat(ets) if len(ets) else torch.empty(0,3,dtype=torch.long)
    etime = torch.cat(etime) if len(etime) else torch.empty(0)
    cal = torch.cat(cal) if len(cal) else torch.empty(0,3)
    return dict(profile=prof, profile_mask=prof_mask, profile_time=ptime, lifelong=ll, lifelong_mask=ll_mask, lifelong_time=ltime, event_tokens=etoks, event_offsets=eoffs, event_user=torch.tensor(eu), event_time=etime, cal=cal, history_offsets=torch.tensor(hoffs), uids=[o['uid'] for o in recs])

In [ ]:
df = pl.read_parquet(shards[0])
uids = df.select('user_id').unique(maintain_order=True).head(2)['user_id'].to_list()
recs = [user_rec(df.filter(pl.col('user_id') == u)) for u in uids]
b = pack_batch(recs)

test_eq(b['profile'].shape[0], 2)
test_eq(b['lifelong'].shape[0], 2)
test_eq(b['event_offsets'][0].item(), 0)
test_eq(b['event_offsets'][-1].item(), len(b['event_tokens']))
test_eq(len(b['event_user']), len(b['event_offsets'])-1)
test_eq(len(b['event_time']), len(b['event_user']))
test_eq(b['cal'].shape, (len(b['event_user']),3))
test_eq(b['history_offsets'][0].item(), 0)
test_eq(b['history_offsets'][-1].item(), len(b['event_user']) + len(recs))
test_eq(b['uids'], uids)

b['profile'].shape,b['lifelong'].shape,b['event_tokens'].shape,b['event_offsets'].shape,b['event_user'].shape,b['history_offsets']

(torch.Size([2, 4, 3]),
 torch.Size([2, 0, 3]),
 torch.Size([530, 3]),
 torch.Size([266]),
 torch.Size([265]),
 tensor([  0,  55, 267]))

In [ ]:
#| export
class PRAGMADataLoader(IterableDataset):
    def __init__(self, shards, max_tokens=max_tokens, shuffle=False):
        store_attr('shards,max_tokens,shuffle')
    def __iter__(self):
        paths = list(self.shards)
        if self.shuffle: paths = L(paths).shuffle()
        batch,ntoks = [],0
        for p in paths:
            df = pl.read_parquet(p)
            for g in df.partition_by('user_id', maintain_order=True):
                r = user_rec(g)
                n = rec_ntoks(r)
                if batch and ntoks+n>self.max_tokens:
                    yield pack_batch(batch)
                    batch,ntoks = [],0
                batch.append(r); ntoks += n
        if batch: yield pack_batch(batch)

In [ ]:
#| export
@patch
def __len__(self:PRAGMADataLoader):
    if hasattr(self, '_len'): return self._len
    batch,ntoks,n_batches = [],0,0
    for p in self.shards:
        df = pl.read_parquet(p)
        for g in df.partition_by('user_id', maintain_order=True):
            r = user_rec(g)
            n = rec_ntoks(r)
            if batch and ntoks+n>self.max_tokens:
                n_batches += 1
                batch,ntoks = [],0
            batch.append(r); ntoks += n
    self._len = n_batches + bool(batch)
    return self._len

In [ ]:
dl = PRAGMADataLoader(shards)
len(dl), sum(1 for _ in dl)

(18, 18)

In [ ]:
dl = DataLoader(PRAGMADataLoader(shards, max_tokens=12000), batch_size=None)
b = first(dl)
{k:(v.shape if hasattr(v,'shape') else len(v)) for k,v in b.items()}

{'profile': torch.Size([55, 4, 3]),
 'profile_mask': torch.Size([55, 4]),
 'profile_time': torch.Size([55, 4]),
 'lifelong': torch.Size([55, 0, 3]),
 'lifelong_mask': torch.Size([55, 0]),
 'lifelong_time': torch.Size([55, 0]),
 'event_tokens': torch.Size([11388, 3]),
 'event_offsets': torch.Size([5695]),
 'event_user': torch.Size([5694]),
 'event_time': torch.Size([5694]),
 'cal': torch.Size([5694, 3]),
 'history_offsets': torch.Size([56]),
 'uids': 55}

In [ ]:
for i,b in zip(range(3), dl):
    test_eq(b['event_offsets'][0].item(), 0)
    test_eq(b['event_offsets'][-1].item(), len(b['event_tokens']))
    test_eq(len(b['event_user']), len(b['event_offsets'])-1)
    test_eq(len(b['event_time']), len(b['event_user']))
    test_eq(b['cal'].shape, (len(b['event_user']),3))
    test_eq(b['history_offsets'][0].item(), 0)
    test_eq(b['history_offsets'][-1].item(), len(b['event_user']) + len(b['uids']))
    test_eq(b['profile'].shape[:2], b['profile_mask'].shape)
    test_eq(b['lifelong'].shape[:2], b['lifelong_mask'].shape)
    print(i, len(b['uids']), len(b['event_user']), len(b['event_tokens']))

0 55 5694 11388


1 39 5918 11836


2 55 5865 11730


## Masking, splits, and dataloader wrapper

Assumptions:
- MLM labels reconstruct value IDs only, using `event_tokens[:,1]`.
- Non-masked labels use `-100` so PyTorch loss can ignore them.
- Batches are built by token budget rather than fixed user count.
- Multi-worker loading splits shard paths across workers to avoid duplicate data.

## MLM masking assumptions

We mask only event value tokens, not profile or lifelong tokens.

For each user record, selected mask positions come from the union of three sources:

- **Token-level masking:** each event token is selected independently with probability `p_tok`.
- **Event-level masking:** each event is selected independently with probability `p_event`, and all value tokens in that event are selected.
- **Semantic key-level masking:** keys are selected independently per user with probability `p_key`, and all event values for those keys are selected for that user.

After positions are selected:

- selected positions become MLM targets using their original `val_id`
- most selected input values are replaced with `[MASK]`
- a small fraction are replaced with `[UNK]`
- `[UNK]` positions are excluded from the loss by setting their label to `-100`

So the effective target mask is:

This follows the paper’s intent while treating key-level masking as sample-local: selected keys affect only the user record where they were sampled, not other users in the same batch.

In [ ]:
#| export
def _vocab_id(vocab, *names):
    for n in names:
        if n in vocab: return vocab[n]
    raise KeyError(f"missing vocab token: {names}")

In [ ]:
#| export
def mask_batch(b, tok, p_tok=.15, p_event=.10, p_key=.10, p_unk=.05, ignore=-100):
    x = b['event_tokens'].clone()
    n = len(x)
    y = torch.full((n,), ignore, dtype=torch.long)
    if n == 0:
        b = b.copy()
        b['event_tokens'],b['event_labels'],b['mlm_mask'] = x,y,y != ignore
        return b

    mask_id = _vocab_id(tok.val_vocab, '[MASK]', '<MASK>', '[mask]', '<mask>')
    unk_id = _vocab_id(tok.val_vocab, '[UNK]', '<UNK>', '[unk]', '<unk>')
    lens = b['event_offsets'][1:] - b['event_offsets'][:-1]
    tev = torch.repeat_interleave(torch.arange(len(lens)), lens)
    tu = b['event_user'][tev]
    nk = max(tok.key_vocab.values()) + 1

    m = torch.rand(n) < p_tok
    m |= (torch.rand(len(lens)) < p_event)[tev]
    m |= (torch.rand(len(b['uids']), nk) < p_key)[tu, x[:,0]]

    y[m] = x[m,1]
    u = m & (torch.rand(n) < p_unk)
    x[m & ~u,1] = mask_id
    x[u,1] = unk_id
    y[u] = ignore

    b = b.copy()
    b['event_tokens'],b['event_labels'],b['mlm_mask'] = x,y,y != ignore
    return b

In [ ]:
#| export
def split_shards(shards, valid_pct=.2):
    n = max(1, round(len(shards)*valid_pct))
    return shards[:-n],shards[-n:]

In [ ]:
#| export
class PRAGMADataLoader(IterableDataset):
    def __init__(self, shards, max_tokens=max_tokens, shuffle=False, tok=None, mask=False):
        store_attr('shards,max_tokens,shuffle,tok,mask')
    def _paths(self):
        paths = list(self.shards)
        if self.shuffle: paths = L(paths).shuffle()
        wi = torch.utils.data.get_worker_info()
        return paths if wi is None else paths[wi.id::wi.num_workers]
    def __iter__(self):
        batch,ntoks = [],0
        for p in self._paths():
            df = pl.read_parquet(p)
            for g in df.partition_by('user_id', maintain_order=True):
                r = user_rec(g); n = rec_ntoks(r)
                if batch and ntoks+n>self.max_tokens:
                    b = pack_batch(batch)
                    yield mask_batch(b, self.tok) if self.mask else b
                    batch,ntoks = [],0
                batch.append(r); ntoks += n
        if batch:
            b = pack_batch(batch)
            yield mask_batch(b, self.tok) if self.mask else b

In [ ]:
#| export
def pragma_dl(shards, max_tokens=max_tokens, shuffle=False, tok=None, mask=False, num_workers=0):
    return DataLoader(PRAGMADataLoader(shards, max_tokens, shuffle, tok, mask), batch_size=None, num_workers=num_workers)

In [ ]:
b = first(pragma_dl(shards, max_tokens=50000, shuffle=True, tok=tok, mask=True))

test_eq(b['event_labels'].shape, (len(b['event_tokens']),))
test_eq(b['mlm_mask'].shape, (len(b['event_tokens']),))
test_eq((b['event_labels'][~b['mlm_mask']] == -100).all().item(), True)
test_eq((b['event_labels'][b['mlm_mask']] != -100).all().item(), True)

b['mlm_mask'].float().mean(), b['event_tokens'][:5], b['event_labels'][:5]

(tensor(0.3076),
 tensor([[  7, 902,   0],
         [  9,   1,   0],
         [  7, 914,   0],
         [  9,   1,   0],
         [  7, 867,   0]]),
 tensor([-100, 1507, -100, 1507, -100]))

In [ ]:
#| export
def pragma_dls(out, valid_pct=.2, max_tokens=max_tokens, mask=True, num_workers=0):
    out = Path(out)
    tok = Tokenizer.load(out/'tokenizer.json')
    trn,val = split_shards(sorted(out.glob('shard_*.parquet')), valid_pct)
    return tok,pragma_dl(trn, max_tokens, True, tok, mask, num_workers),pragma_dl(val, max_tokens, False, tok, mask, num_workers)

## Final dataloader sanity check

Assumptions:
- A valid batch must have consistent offsets, event counts, history offsets, masks, and MLM label shapes.
- These checks validate the data contract expected by the next model-building notebook.

In [ ]:
tok,trn_dl,val_dl = pragma_dls(out, max_tokens=12000, num_workers=0)
b = first(trn_dl)

test_eq(b['event_offsets'][0].item(), 0)
test_eq(b['event_offsets'][-1].item(), len(b['event_tokens']))
test_eq(b['history_offsets'][-1].item(), len(b['event_user']) + len(b['uids']))
test_eq(b['event_labels'].shape, b['mlm_mask'].shape)
test_eq(b['event_labels'].shape, (len(b['event_tokens']),))
test_eq((b['event_labels'][~b['mlm_mask']] == -100).all().item(), True)
test_eq((b['event_labels'][b['mlm_mask']] != -100).all().item(), True)

{k:(v.shape if hasattr(v,'shape') else len(v)) for k,v in b.items()}

{'profile': torch.Size([54, 4, 3]),
 'profile_mask': torch.Size([54, 4]),
 'profile_time': torch.Size([54, 4]),
 'lifelong': torch.Size([54, 0, 3]),
 'lifelong_mask': torch.Size([54, 0]),
 'lifelong_time': torch.Size([54, 0]),
 'event_tokens': torch.Size([11776, 3]),
 'event_offsets': torch.Size([5889]),
 'event_user': torch.Size([5888]),
 'event_time': torch.Size([5888]),
 'cal': torch.Size([5888, 3]),
 'history_offsets': torch.Size([55]),
 'uids': 54,
 'event_labels': torch.Size([11776]),
 'mlm_mask': torch.Size([11776])}

In [ ]:
b = first(pragma_dl(shards, max_tokens=12000, shuffle=True, tok=tok, mask=True))
u = torch.randint(len(b['uids']), ()).item()
evs = (b['event_user'] == u).nonzero().flatten()
e = evs[torch.randint(len(evs), ()).item()].item()
s,t = b['event_offsets'][e].item(),b['event_offsets'][e+1].item()
hist_s,hist_t = b['history_offsets'][u].item(),b['history_offsets'][u+1].item()

test_eq(b['event_offsets'][0].item(), 0)
test_eq(b['event_offsets'][-1].item(), len(b['event_tokens']))
test_eq(b['event_user'][e].item(), u)
test_eq(t-s > 0, True)
test_eq(hist_t-hist_s, 1+len(evs))
test_eq(b['event_tokens'][s:t].shape[1], 3)
test_eq(b['event_labels'].shape, (len(b['event_tokens']),))
test_eq(b['mlm_mask'].shape, (len(b['event_tokens']),))

b['uids'][u], u, e, s, t, b['event_tokens'][s:t], b['event_labels'][s:t], b['mlm_mask'][s:t]

(68,
 16,
 1185,
 2370,
 2372,
 tensor([[   7,    1,    0],
         [   9, 1507,    0]]),
 tensor([1460, -100]),
 tensor([ True, False]))

In [ ]:
uid = 1
df1 = pl.concat([pl.read_parquet(p).filter(pl.col('user_id') == uid) for p in shards])
rec1 = user_rec(df1)
b1 = pack_batch([rec1])

test_eq(b1['uids'], [uid])
test_eq(b1['event_offsets'][0].item(), 0)
test_eq(b1['event_offsets'][-1].item(), len(b1['event_tokens']))
test_eq(len(b1['event_user']), len(rec1['events']))
test_eq(b1['history_offsets'].tolist(), [0, len(rec1['events'])+1])

ev = df1.filter(pl.col('event_idx') >= 0).sort('event_idx')
ev0 = ev.filter(pl.col('event_idx') == ev['event_idx'][0]).head(max_event_toks)
x0,_ = _tok(ev0)
s,t = b1['event_offsets'][0].item(),b1['event_offsets'][1].item()

test_eq(b1['event_tokens'][s:t], x0)
test_eq(b1['event_user'][0].item(), 0)
test_eq(b1['cal'][0], torch.tensor(ev0.select('hour','dow','dom').row(0), dtype=torch.float32))

uid, df1.shape, b1['profile'].shape, b1['event_tokens'].shape, b1['event_offsets'].shape, b1['history_offsets'], b1['event_tokens'][s:t]

(1,
 (548, 15),
 torch.Size([1, 4, 3]),
 torch.Size([544, 3]),
 torch.Size([273]),
 tensor([  0, 273]),
 tensor([[   7,  902,    0],
         [   9, 1507,    0]]))

In [ ]:
#| export
batch_fields = 'profile profile_mask profile_time lifelong lifelong_mask lifelong_time event_tokens event_offsets event_user event_time cal history_offsets uids event_labels mlm_mask'.split()
special_toks = dict(usr='[USR]', evt='[EVT]')

def batch_shapes(b): return {k:(tuple(v.shape) if hasattr(v,'shape') else len(v)) for k,v in b.items()}

In [ ]:
tok,trn_dl,val_dl = pragma_dls(out, max_tokens=12000, num_workers=0)
b = first(trn_dl)

test_eq(all(k in b for k in batch_fields), True)
test_eq(b['event_offsets'][-1].item(), len(b['event_tokens']))
test_eq(len(b['event_user']), len(b['event_offsets'])-1)
test_eq(len(b['event_time']), len(b['event_user']))
test_eq(b['cal'].shape, (len(b['event_user']),3))
test_eq(b['history_offsets'][-1].item(), len(b['event_user'])+len(b['uids']))
test_eq(b['event_labels'].shape, b['mlm_mask'].shape)
test_eq(b['event_labels'].shape, (len(b['event_tokens']),))

batch_shapes(b)

{'profile': (60, 4, 3),
 'profile_mask': (60, 4),
 'profile_time': (60, 4),
 'lifelong': (60, 0, 3),
 'lifelong_mask': (60, 0),
 'lifelong_time': (60, 0),
 'event_tokens': (11602, 3),
 'event_offsets': (5802,),
 'event_user': (5801,),
 'event_time': (5801,),
 'cal': (5801, 3),
 'history_offsets': (61,),
 'uids': 60,
 'event_labels': (11602,),
 'mlm_mask': (11602,)}

## Dataloader contract

The model receives padded profile/lifelong tensors and packed event tensors.

Special tokens are added by the model, not the dataloader:
- profile encoder prepends `[USR]`
- event encoder prepends `[EVT]` to each event
- history encoder receives `[USR] + [EVT]` event representations

`event_offsets` maps packed event tokens to events.
`event_user` maps events to batch users.
`history_offsets` maps history tokens to users.

MLM labels reconstruct event value IDs only. Non-masked labels are `-100`.

In [ ]:
#| export
max_profile_toks,max_lifelong_toks,max_events,max_event_toks,max_tokens = 200,64,6500,24,12000

In [ ]:
test_eq(special_toks, dict(usr='[USR]', evt='[EVT]'))
test_eq(all(k in b for k in batch_fields), True)
batch_shapes(b)

{'profile': (60, 4, 3),
 'profile_mask': (60, 4),
 'profile_time': (60, 4),
 'lifelong': (60, 0, 3),
 'lifelong_mask': (60, 0),
 'lifelong_time': (60, 0),
 'event_tokens': (11602, 3),
 'event_offsets': (5802,),
 'event_user': (5801,),
 'event_time': (5801,),
 'cal': (5801, 3),
 'history_offsets': (61,),
 'uids': 60,
 'event_labels': (11602,),
 'mlm_mask': (11602,)}

In [ ]:
uid = b['uids'][0]
dfu = pl.concat([pl.read_parquet(p).filter(pl.col('user_id') == uid) for p in shards])
ev = dfu.filter(pl.col('event_idx') >= 0).sort('event_idx')
kept = ev.select('event_idx').unique(maintain_order=True).tail(max_events)['event_idx'].to_list()
rec = user_rec(dfu)

test_eq(len(rec['events']), len(kept))
test_eq(len(rec['events']) <= max_events, True)
test_eq(kept, sorted(kept))
len(kept), kept[:3], kept[-3:]

(62, [0, 1, 2], [59, 60, 61])

In [ ]:
seen = []
for b in pragma_dl(shards, max_tokens=12000, shuffle=False, tok=tok, mask=False): seen += b['uids']

expected = set(pl.scan_parquet(shards).select('user_id').unique().collect()['user_id'])
seen = set(seen)

test_eq(seen, expected)
len(seen), len(expected)

(943, 943)

In [ ]:
import nbdev; nbdev.nbdev_export()

In [ ]:
from fastpragma.dataloader import *
batch_fields, max_tokens

(['profile',
  'profile_mask',
  'profile_time',
  'lifelong',
  'lifelong_mask',
  'lifelong_time',
  'event_tokens',
  'event_offsets',
  'event_user',
  'event_time',
  'cal',
  'history_offsets',
  'uids',
  'event_labels',
  'mlm_mask'],
 12000)

summarize the assumptions made here and to be kept in mind while creating the model

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Key assumptions from this dataloader notebook to carry into the model:

- **Input token format**
  - Every key-value token is a triple:

    ```python
    key_id,val_id,val_pos
    ```

  - The model should embed `key_id`, `val_id`, and probably use `val_pos` for within-field position.

- **Profile and lifelong are padded**
  - `profile`: shape `(bs, max_profile_len, 3)`
  - `lifelong`: shape `(bs, max_lifelong_len, 3)`
  - masks tell which rows are real:

    ```python
    profile_mask
    lifelong_mask
    ```

- **Events are packed, not padded**
  - `event_tokens`: shape `(total_event_tokens, 3)`
  - `event_offsets`: length `n_events+1`
  - event `i` lives at:

    ```python
    event_tokens[event_offsets[i]:event_offsets[i+1]]
    ```

- **Events know their users**
  - `event_user`: length `n_events`
  - `event_user[i]` gives the batch user index for event `i`.

- **History encoder layout**
  - `history_offsets` has length `bs+1`
  - each user gets:

    ```text
    1 [USR] token + one [EVT] token per event
    ```

  - so the model must build history sequences from profile representation plus event representations using these offsets.

- **Special tokens are model-side**
  - dataloader does not insert `[USR]` or `[EVT]`
  - model should prepend/add them internally:
    - `[USR]` before profile encoding
    - `[EVT]` before each event encoding

- **Time features**
  - `profile_time` and `lifelong_time` align with profile/lifelong token rows.
  - `event_time` has one value per event, not per event token.
  - `cal` has one `(hour,dow,dom)` row per event.

- **Truncation**
  - profile rows capped by `max_profile_toks`
  - lifelong rows capped by `max_lifelong_toks`
  - latest events capped by `max_events`
  - each event capped by `max_event_toks`

- **Batching**
  - batches are formed by approximate token budget, not fixed user count
  - batch size varies
  - model code should not assume constant `bs` or constant number of events.

- **MLM labels**
  - masking is currently for event value IDs only:

    ```python
    event_labels ~ event_tokens[:,1]
    ```

  - non-target positions are `-100`
  - loss should ignore `-100`.

- **Model responsibility**
  - convert packed event tokens into per-event embeddings
  - convert padded profile/lifelong tokens into one user/profile embedding
  - combine `[USR] + [EVT]...` per user for the history encoder
  - use offsets/masks rather than assuming rectangular event tensors.